# 05 - Capel-wide simulation bank driver

This notebook runs **N simulations** using `generate_sims_capel_wide.py` and stores outputs in:

`/mnt/my-ssd/Data_for_IceCube_Population_Project/Sims/Capel_wide`


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from icecube_population_project import (
    SmokeSimConfig,
    TightPriorConfig,
    generate_smoke_sim_bank,
    read_real_map_metadata,
    run_sanity_checks,
)


: 

In [ ]:
DATA_ROOT = Path('/mnt/my-ssd/Data_for_IceCube_Population_Project')
REAL_MAPS_NPZ = DATA_ROOT / 'Results' / 'icecube10y_counts_maps.npz'
OUTDIR = DATA_ROOT / 'Sims' / 'Capel_wide'

N_SIMS = 200_000
SEED = 4698
OVERWRITE = True
RUN_SANITY_CHECK = True

real_meta = read_real_map_metadata(REAL_MAPS_NPZ)

sim_cfg = SmokeSimConfig(
    n_sims=N_SIMS,
    seed=SEED,
    nside=128,
    nest=True,  # recommended for DeepHpx pooling/unpooling
    output_format='zarr',
    zarr_chunk_sims=1,
    truth_store=True,
    truth_topk=64,
    truth_min_expected_counts=0.0,
    time_years=float(real_meta['livetime_years_total']),
    energy_edges_GeV=tuple(np.asarray(real_meta['E_edges_GeV'], dtype=float).tolist()),
    real_maps_npz=str(REAL_MAPS_NPZ),
    enforce_match_real_maps=True,
    sources_to_draw=2000,
    energy_grid_per_decade=6,
    reco_batch_size=200_000,
    astro_psf_mode='fixed_event',
    fixed_psf_sigma_deg=1.0,
    atmo_psf_mode='none',
    conventional_model='honda2006',
    include_prompt=False,
)

prior_cfg = TightPriorConfig()

print('REAL_MAPS_NPZ :', REAL_MAPS_NPZ)
print('OUTDIR        :', OUTDIR)
print('N simulations :', sim_cfg.n_sims)
print('NSIDE / NPIX  :', sim_cfg.nside, 12 * sim_cfg.nside * sim_cfg.nside)
print('NEST ordering :', sim_cfg.nest)
print('Output format :', sim_cfg.output_format)
print('Store truth   :', sim_cfg.truth_store)
print('Livetime [yr] :', sim_cfg.time_years)
print('Energy edges  :', np.asarray(sim_cfg.energy_edges_GeV))


REAL_MAPS_NPZ : /mnt/my-ssd/Data_for_IceCube_Population_Project/Results/icecube10y_counts_maps.npz
OUTDIR        : /mnt/my-ssd/Data_for_IceCube_Population_Project/Sims/Capel_wide
N simulations : 50000
NSIDE / NPIX  : 128 196608
NEST ordering : True
Output format : zarr
Store truth   : True
Livetime [yr] : 7.8061182882671565
Energy edges  : [   10000.            39810.71705535   158489.31924611   630957.34448019
  2511886.43150958 10000000.        ]


In [3]:
if RUN_SANITY_CHECK:
    report = run_sanity_checks(
        energy_edges_GeV=np.asarray(sim_cfg.energy_edges_GeV),
        nside_quick=16,
        sources_to_draw_quick=300,
        time_years_quick=1.0,
        rng_seed=123,
    )
    report
else:
    print('Skipping run_sanity_checks (RUN_SANITY_CHECK=False).')


In [4]:
outputs = generate_smoke_sim_bank(
    outdir=OUTDIR,
    sim_cfg=sim_cfg,
    prior=prior_cfg,
    overwrite=OVERWRITE,
)
outputs


{'maps': PosixPath('/mnt/my-ssd/Data_for_IceCube_Population_Project/Sims/Capel_wide/maps_counts.zarr'),
 'maps_counts': PosixPath('/mnt/my-ssd/Data_for_IceCube_Population_Project/Sims/Capel_wide/maps_counts.zarr'),
 'theta_csv': PosixPath('/mnt/my-ssd/Data_for_IceCube_Population_Project/Sims/Capel_wide/theta.csv'),
 'meta': PosixPath('/mnt/my-ssd/Data_for_IceCube_Population_Project/Sims/Capel_wide/meta.json'),
 'truth_hotspot_sparse': PosixPath('/mnt/my-ssd/Data_for_IceCube_Population_Project/Sims/Capel_wide/truth_hotspot_sparse.npz')}

In [5]:
maps_path = Path(outputs['maps_counts'])
if maps_path.suffix == '.zarr' or maps_path.is_dir():
    import zarr

    zobj = zarr.open(str(maps_path), mode='r')
    maps = zobj['maps_counts'] if hasattr(zobj, 'keys') and 'maps_counts' in zobj else zobj
else:
    maps = np.load(maps_path, mmap_mode='r')

theta = pd.read_csv(outputs['theta_csv'])

print('maps path :', maps_path)
print('maps shape:', maps.shape, 'dtype:', maps.dtype)
print('theta shape:', theta.shape)
display(theta.head())


maps path : /mnt/my-ssd/Data_for_IceCube_Population_Project/Sims/Capel_wide/maps_counts.zarr
maps shape: (50000, 5, 196608) dtype: uint32
theta shape: (50000, 19)


,sim_id,n0_Mpc3,log10_n0,L_TeV_s,log10_L_TeV_s,L_erg_s,gamma,p1,p2,zc,seed_catalog,seed_astro,seed_atmo,expected_astro_total,expected_atmo_total,astro_events_total,astro_events_kept,atmo_events_total,atmo_events_kept
0,0,1.901542e-07,-6.720894,7.791788e+40,40.891637,1.248382e+41,1.898515,20.454945,21.805279,1.593474,10000000,20000000,30000000,85.281255,20108.689906,91,24,20222,1422
1,1,2.439852e-09,-8.612636,4.874551e+40,40.687935,7.809892e+40,2.622357,19.826831,24.611433,1.447864,10000001,20000001,30000001,0.038740,20108.689906,0,0,20213,1409
2,2,4.145539e-10,-9.382419,3.951210e+40,40.596730,6.330536e+40,2.232633,19.458866,21.322256,1.373277,10000002,20000002,30000002,0.013204,20108.689906,0,0,19994,1407
3,3,5.655875e-10,-9.247500,2.985914e+42,42.475077,4.783962e+42,1.663126,19.480725,20.098929,1.293870,10000003,20000003,30000003,1.133381,20108.689906,1,1,20221,1462
4,4,1.429956e-09,-8.844677,1.102977e+43,43.042566,1.767164e+43,2.011775,19.763536,23.111747,1.757152,10000004,20000004,30000004,47.088601,20108.689906,39,9,19891,1360


## Minimal, practical plan for the NPE + DeepHpx pipeline comparisons against Capel+2020

### Keep `sources_to_draw=2000` for now, but “lock it in” via a convergence test

For the “independent inference pipeline with similar sensitivity,” the minimal safe approach is:

1. **Treat `sources_to_draw` as a numerical accuracy knob.**
2. Pick the smallest value that makes the **map statistics converge** (within tolerance) over the *parameter region you care about*.

**Minimal convergence harness (fast to implement):**

* Choose 3–5 representative parameter points ((n_0,L,\gamma,\theta)) spanning the tightened prior (e.g., one “dense faint” and one “rare bright” regime).
* For each point, generate (say) 20 skies at:

  * `sources_to_draw = 2k, 5k, 20k` (same random seed sequence except for catalog draw)
* Compare a few summary diagnostics per energy bin:

  * pixel-count histogram moments (mean/var of log1p counts)
  * extreme tail: 99.9th percentile of pixel counts
  * optional: low-ℓ angular power (C_\ell) (even ℓ=1..20 is enough)

If those summaries change by <~5% going from 5k → 20k, you’re converged at 5k; if 2k → 5k already converges, keep 2k.

This is the cheapest way to ensure your map-based inference isn’t accidentally limited by “2000 super-sources.”

### Why this works for our current goal

Capel’s constraints are driven by the relationship between diffuse flux and point-source detectability, which is sensitive to anisotropy/tails. ([Physical Review Journals][1])
The convergence test makes sure you’re not suppressing those tails artificially.

[1]: https://link.aps.org/pdf/10.1103/PhysRevD.101.123017 "Bayesian constraints on the astrophysical neutrino source population from IceCube data"
[2]: https://arxiv.org/pdf/2505.02906 "A robust neural determination of the source-count distribution of the Fermi-LAT sky at high latitudes"
